# spaCy på Folketingets debatter: hvad forsvinder, og hvad er komsekvenserne af det?

**Social Data Science 1: lektion 7**

I lektion 6 gik vi gennem Denny & Spirlings syv operationer én ad gangen og talte, hvor mange
**tokens** (forekomster) og **typer** (unikke ord) der var tilbage efter hvert trin. Pointen
var, at hver enkelt punkt er et valg om, hvad der tæller som
*det samme ord*.

Her gør vi det samme på debatterne fra `ft_lovforslag.csv`, med **spaCy**, der
afgør, hvad der er tegnsætning, tal, stopord og grundform. For hvert trin spørger vi:

1. **Hvor meget** forsvinder? Tokens, typer eller begge?
2. **Hvad** forsvinder? 
3. **Hvad betyder det** for en analyse af politiske debatter?

<br>

| Trin | Operation (Denny & Spirling) | spaCy-egenskab |
|---|---|---|
| 1 | **L** små bogstaver | `.str.lower()` |
| 2 | **P** fjern tegnsætning | `is_punct` |
| 3 | **N** fjern tal | `like_num` |
| 4 | **W** fjern stopord | `is_stop` |
| 5 | **S** grundform: lemmatisering i stedet for stemming | `lemma_` |
| 6 | *(ikke blandt de syv)* behold kun visse ordklasser | `pos_` |
| 7 | **I** fjern sjældne ord | optælling |
| 8 | **3** ord der hører sammen: navne som ét token | `merge_entities` |


---

## 1. Opsætning

Modellen hentes én gang i terminalen:

```
pip install spacy
python -m spacy download da_core_news_sm
```

In [1]:
import time
import string

import pandas as pd
import spacy

nlp = spacy.load("da_core_news_sm")

/Users/jeppefl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Vi skal bruge den samme lille funktion flere gange. Funktionen laver et `Doc` om til en tabel med **én række per token**. Hver kolonne er en af de egenskaber, spaCy har fundet.

In [2]:
def tokentabel(doc):
    return pd.DataFrame({
        "tekst":      [t.text for t in doc],
        "lemma":      [t.lemma_ for t in doc],
        "ordklasse":  [t.pos_ for t in doc],
        "tegn":       [t.is_punct for t in doc],
        "tal":        [t.like_num for t in doc],
        "stopord":    [t.is_stop for t in doc],
        "enhed":      [t.ent_type_ for t in doc],
        "mellemrum":  [t.is_space for t in doc],
    })

---

## 2. Et mikroeksempel

Her er nogle kunstige sætninger som skal ligne Folketingets sprog: paragraffer, beløb, lovforslagsnumre, partinavne og nægtelser.

In [3]:
mikro = [
    "Regeringen vil sænke skatten på arbejde, fordi de danske lønmodtagere betaler for meget.",
    "Efter § 17, stk. 2, skal der betales 10 mia. kr. i 2024.",
    "Tre ministre har nu for 17. gang sagt, at L 12 ikke er klima-venligt.",
    "Vi støtter ikke forslaget.",
    "Venstre og Dansk Folkeparti støtter forslaget, men Enhedslisten er imod.",
    "Ordføreren sad længst til venstre i salen.",
]

mikro_tokens = tokentabel(nlp(" ".join(mikro)))

**Kig efter i viewer:** Gå kolonnerne igennem én ad gangen. I `da_core_news_sm` 3.8.0, var
dette nogle af resultaterne:

- `tegn`: `§` regnes som tegnsætning.
- `tal`: både `17`, `2024`, `17.` og ordet `Tre` regnes som tal. I `L 12` er `12` et tal,
  men `L` er det ikke.
- `stopord`: `ikke` og `imod` står på stopordslisten.
- `lemma`: *skatten* får lemmaet *skatt*, og *Ordføreren* og *betales* bliver stående uændret.
  *Venstre* (partiet) og *venstre* (retningen) får samme lemma.
- `enhed`: *Venstre* og *Dansk Folkeparti* er organisationer, men *Enhedslisten* er en person.

Hvad er der tilbage, hvis vi fjerner tegnsætning, tal og stopord, bruger lemmaer og kun
beholder navneord, egennavne og tillægsord, præcis som i lektion 6?

### 2.1 Manuelle rettelser

Mikroeksemplet viste fejl, som vi kan se er forkerte for debatter i Folketinget. spaCy lader
os rette dem, men hver type fejl rettes på sin egen måde:

| Fejl | Hvor den sidder | Hvordan den rettes |
|---|---|---|
| *Enhedslisten* er en person | `ner` | en `entity_ruler` med vores egne navne |
| *skatten* får lemmaet *skatt* | `lemmatizer` | en regel i `attribute_ruler` |
| `§` er tegnsætning, `Tre` er et tal, `ikke` er et stopord | ordforrådet (`vocab`) | sæt egenskaben direkte |

Vi laver en ny pipeline, `nlp_rettet`, så `nlp` stadig viser modellens egne valg i resten af
notebooken.

In [4]:
nlp_rettet = spacy.load("da_core_news_sm")

**Navngivne enheder.** En `entity_ruler` finder præcis de navne, vi giver den. `before="ner"`
sætter den ind før modellens eget gæt, så reglen vinder.

In [5]:
ruler = nlp_rettet.add_pipe("entity_ruler", before="ner")
ruler.add_patterns([{"label": "ORG", "pattern": "Enhedslisten"}])

**Lemmaer.** I den danske pipeline kører `attribute_ruler` efter `lemmatizer`, så en regel her
overskriver modellens lemma. `LOWER` betyder, at reglen rammer ordet uanset store og små
bogstaver.

In [6]:
ar = nlp_rettet.get_pipe("attribute_ruler")
ar.add([[{"LOWER": "skatten"}]], {"LEMMA": "skat"})
ar.add([[{"LOWER": "ordføreren"}]], {"LEMMA": "ordfører"})

**Tegnsætning, tal og stopord.** Det er egenskaber ved selve ordet i modellens ordforråd, så vi
sætter dem direkte. Hver stavemåde er sit eget opslag: `Tre` og `tre` skal rettes hver for sig,
og det samme gælder `Ikke` i starten af en sætning.

In [7]:
nlp_rettet.vocab["§"].is_punct = False

for form in ["tre", "Tre"]:
    nlp_rettet.vocab[form].like_num = False

for form in ["ikke", "Ikke", "imod", "Imod"]:
    nlp_rettet.vocab[form].is_stop = False

Uden ordklassefilteret overlever `§`, `tre`, `ikke` og `imod`. **Med** filteret forsvinder de
alligevel, fordi de har ordklasserne `SYM`, `NUM` og `ADV`. Rettelserne i ét trin hjælper ikke,
hvis et senere trin fjerner de samme ord af en anden grund. Kæden skal ses som en helhed.

> Hver rettelse er også et valg. Vi har besluttet, at `§` er et ord, at *ikke* er indhold, og
> at *Enhedslisten* er en organisation. Det er rigtigt for debatter i Folketinget, men det er
> vores regel, ikke modellens viden. Og vi har kun rettet de fejl, vi har set.

### Samme mikroeksempel igen, nu med rettelserne:

In [9]:
mikro_tokens = tokentabel(nlp_rettet(" ".join(mikro)))

In [10]:
tilbage = mikro_tokens[~mikro_tokens["tegn"]
                       & ~mikro_tokens["tal"]
                       & ~mikro_tokens["stopord"]
                       & mikro_tokens["ordklasse"].isin(["NOUN", "PROPN", "ADJ"])]

print(tilbage["lemma"].str.lower().tolist())

['regering', 'skat', 'arbejde', 'dansk', 'lønmodtager', 'stk.', 'milliard', 'krone', 'minister', 'gang', 'l', 'klima-venlig', 'forslag', 'venstre', 'dansk', 'folkeparti', 'forslag', 'enhedslisten', 'ordfører', 'venstre', 'sal']


---

## 3. Kør spaCy på et udsnit af debatterne

### 3.1 Udsnittet

En debat er typisk 29.000 tegn, og der er over 3.000 af dem. Vi tager 100 tilfældige debatter. `random_state=1` gør, at vi alle får det samme udsnit.

In [11]:
lovforslag = pd.read_csv("ft_lovforslag.csv")

udsnit = lovforslag.sample(100, random_state=1).copy()
udsnit["tekst"] = udsnit["tekst"]

print(udsnit["omraade"].value_counts())

omraade
Andet             47
Retsvæsen         11
Skat              11
Miljø og klima    10
Beskæftigelse     10
Uddannelse         6
Udlændinge         5
Name: count, dtype: int64


### 3.2 Første konsekvens: tid

I lektion 6 kørte reglerne på sekunder. Vi tager for eksemplets skyld tid på spaCy, når den kører på vores udsnit.

In [12]:
start = time.perf_counter()

udsnit["doc"] = list(nlp.pipe(udsnit["tekst"], batch_size=50))

sekunder = time.perf_counter() - start
print("Tid for udsnittet:", round(sekunder, 1), "sekunder")

Tid for udsnittet: 93.7 sekunder


Hvor lang tid ville hele korpusset tage? Vi regner med, at tiden vokser med antallet af tegn.

In [13]:
tegn_udsnit = udsnit["tekst"].str.len().sum()
tegn_alt = lovforslag["tekst"].str.len().sum()

print("Tegn i udsnittet:", tegn_udsnit)
print("Tegn i hele korpusset:", tegn_alt)
print("Anslået tid for hele korpusset:", round(sekunder / tegn_udsnit * tegn_alt / 60, 1), "minutter")

Tegn i udsnittet: 4654510
Tegn i hele korpusset: 126830780
Anslået tid for hele korpusset: 42.6 minutter


### 3.3 Én række per token

Samme opskrift som i lektion 6: tekst bliver til *tidy data* med ét ord per række. `.map()`
laver én tabel per debat, og `pd.concat()` sætter dem oven på hinanden.

In [14]:
tabeller = udsnit.set_index("id")["doc"].map(tokentabel)

ord = pd.concat(tabeller.to_dict(), names=["id", "nr"]).reset_index()
ord = ord.merge(udsnit[["id", "omraade"]], on="id", how="left")

spaCy gemmer linjeskift som tokens. Det gjorde `.split()` i lektion 6 ikke, så for at tallene
kan sammenlignes, fjerner vi dem først.

In [15]:
print("Linjeskift og mellemrum:", ord["mellemrum"].sum())

ord = ord[~ord["mellemrum"]].copy()

Linjeskift og mellemrum: 13406


**Kig efter i viewer:** Én række per token i alle 100 debatter, med alt det, spaCy har fundet om hvert token. Resten
af notebooken er filtre og omdannelser af denne ene tabel.

---

## 4. En tabel til kæden

Vi fører regnskab undervejs, ligesom tabellen *Hele kæden* i lektion 6. Kolonnen `form` er ordet, som det ser ud lige nu. Den ændrer sig, når vi laver små bogstaver og lemmaer. 

In [16]:
kaede = pd.DataFrame(columns=["trin", "tokens", "typer"])

def tael(trin, former):
    kaede.loc[len(kaede)] = [trin, len(former), former.nunique()]
    print(kaede)

In [17]:
ord["form"] = ord["tekst"]

tael("Rå spaCy-tokens", ord["form"])

              trin  tokens  typer
0  Rå spaCy-tokens  922388  27535


---

## 5. Kæden, ét trin ad gangen

### 5.1 L: små bogstaver

Ingen tokens forsvinder, men typer gør: `Regeringen` og `regeringen` bliver én type.

In [18]:
ord["form"] = ord["tekst"].str.lower()

tael("+ små bogstaver", ord["form"])

              trin  tokens  typer
0  Rå spaCy-tokens  922388  27535
1  + små bogstaver  922388  26203


**Hvad blev slået sammen?** For hver ny type tæller vi, hvor mange forskellige stavemåder den
dækker over, og hvilke de er.

In [19]:
sammen_l = (ord
            .groupby("form")["tekst"]
            .agg(stavemaader="nunique", eksempler="unique")
            .sort_values("stavemaader", ascending=False)
            .reset_index())

**Kig efter i viewer:** De fleste er bare et ord i starten af en sætning. Men find `venstre`, `konservative` og
`radikale` med søgefeltet. Partinavne og almindelige ord er nu den samme type.

In [20]:
print(ord.loc[ord["form"] == "venstre", "tekst"].value_counts())

tekst
Venstre    1138
venstre       4
Name: count, dtype: int64


**Konsekvens:** I et korpus af debatter i Folketinget er store bogstaver ofte forskellen på et
parti og et ord. Efter dette trin kan en optælling af *venstre* ikke skelne partiet fra
retningen.

### 5.2 P: fjern tegnsætning

I lektion 6 var tegnsætning det største enkeltproblem, fordi `job.`, `job,` og `job` var tre
ord. spaCy har allerede skilt tegnene fra ordene. Her fjerner vi de tokens, spaCy kalder
tegnsætning.

In [21]:
fjernet_p = ord.loc[ord["tegn"], "tekst"].value_counts().reset_index()

ord = ord[~ord["tegn"]]

tael("+ fjern tegnsætning", ord["form"])

                  trin  tokens  typer
0      Rå spaCy-tokens  922388  27535
1      + små bogstaver  922388  26203
2  + fjern tegnsætning  802536  26179


**Kig efter i viewer:** Hvad er fjernet ud over punktummer og kommaer? Står `§` på listen? Hvor mange gange?

**Konsekvens:** I lovgivning er `§` ikke tegnsætning. Det er et ord for *paragraf*. Vi har
ikke skrevet en regel, der fjerner det. Det er modellens egenskab `is_punct`, der har afgjort
det.

### 5.3 N: fjern tal

spaCys `like_num` er bredere end reglen `\d` fra lektion 6, da den også fanger tal skrevet med bogstaver.

In [22]:
fjernet_n = ord.loc[ord["tal"], "tekst"].value_counts().reset_index()

ord = ord[~ord["tal"]]

tael("+ fjern tal", ord["form"])

                  trin  tokens  typer
0      Rå spaCy-tokens  922388  27535
1      + små bogstaver  922388  26203
2  + fjern tegnsætning  802536  26179
3          + fjern tal  776878  25664


**Kig efter i viewer:** Hvilke slags tal er det? Årstal, paragraffer, beløb, lovforslagsnumre, datoer? 

**Konsekvens:** I lektion 6 var spørgsmålet, om tal er støj eller indhold. I debatter om
lovforslag er de ofte indhold: *§ 17*, *2030-målet*, *10 mia. kr.* Samtidig bliver *L 12*
halveret: tallet forsvinder, men `L` bliver stående som et løst bogstav.

In [23]:
print("Tokens der er et løst 'l':", (ord["form"] == "l").sum())

Tokens der er et løst 'l': 146


### 5.4 W: fjern stopord

spaCys danske stopordsliste er færdig og kan læses. Den er længere end den, vi skrev i
lektion 6.

In [24]:
stopord = nlp.Defaults.stop_words

print(len(stopord), "stopord")
print(sorted(stopord))

219 stopord
['af', 'aldrig', 'alene', 'alle', 'allerede', 'alligevel', 'alt', 'altid', 'anden', 'andet', 'andre', 'at', 'bag', 'begge', 'blandt', 'blev', 'blive', 'bliver', 'burde', 'bør', 'da', 'de', 'dem', 'den', 'denne', 'dens', 'der', 'derefter', 'deres', 'derfor', 'derfra', 'deri', 'dermed', 'derpå', 'derved', 'det', 'dette', 'dig', 'din', 'dine', 'disse', 'dog', 'du', 'efter', 'egen', 'eller', 'ellers', 'en', 'end', 'endnu', 'ene', 'eneste', 'enhver', 'ens', 'enten', 'er', 'et', 'flere', 'flest', 'fleste', 'for', 'foran', 'fordi', 'forrige', 'fra', 'få', 'før', 'først', 'gennem', 'gjorde', 'gjort', 'god', 'gør', 'gøre', 'gørende', 'ham', 'han', 'hans', 'har', 'havde', 'have', 'hel', 'heller', 'hen', 'hende', 'hendes', 'henover', 'her', 'herefter', 'heri', 'hermed', 'herpå', 'hun', 'hvad', 'hvem', 'hver', 'hvilke', 'hvilken', 'hvilkes', 'hvis', 'hvor', 'hvordan', 'hvorefter', 'hvorfor', 'hvorfra', 'hvorhen', 'hvori', 'hvorimod', 'hvornår', 'hvorved', 'i', 'igen', 'igennem', 'ikke'

In [25]:
fjernet_w = ord.loc[ord["stopord"], "form"].value_counts().reset_index()

ord = ord[~ord["stopord"]].copy()

tael("+ fjern stopord", ord["form"])

                  trin  tokens  typer
0      Rå spaCy-tokens  922388  27535
1      + små bogstaver  922388  26203
2  + fjern tegnsætning  802536  26179
3          + fjern tal  776878  25664
4      + fjern stopord  312523  25452


Samme mønster som i lektion 6, hvor **mange tokens** forsvinder, men **få typer**. Teksten bliver
kortere, men tabellen bliver ikke meget smallere.

Står nægtelserne på listen? Det kan vi spørge pandas om direkte:

In [26]:
naegtelser = pd.Series(["ikke", "ingen", "aldrig", "imod", "uden"])

print(pd.DataFrame({"ord": naegtelser, "stopord": naegtelser.isin(stopord)}))

      ord  stopord
0    ikke     True
1   ingen     True
2  aldrig     True
3    imod     True
4    uden     True


**Konsekvens:** I en nyhedsartikel er *ikke* måske støj. I en debat i Folketinget er det
forskellen på at støtte et lovforslag og at stemme imod det. Efter dette trin er *Vi støtter
forslaget* og *Vi støtter ikke forslaget* den samme tekst.

Hvor mange nægtelser har vi lige fjernet i udsnittet?

In [27]:
print(fjernet_w[fjernet_w["form"].isin(naegtelser)])

       form  count
11     ikke  10492
93     uden    499
97    ingen    455
103    imod    369
146  aldrig    139


### 5.5 S: lemmatisering i stedet for stemming

Stemming klipper endelser af efter en regel. Lemmatisering slår grundformen op i en model.
Ingen tokens forsvinder, men mange typer bliver slået sammen.

In [28]:
ord["for_lemma"] = ord["form"]
ord["form"] = ord["lemma"].str.lower()

tael("+ lemmatisering", ord["form"])

                  trin  tokens  typer
0      Rå spaCy-tokens  922388  27535
1      + små bogstaver  922388  26203
2  + fjern tegnsætning  802536  26179
3          + fjern tal  776878  25664
4      + fjern stopord  312523  25452
5      + lemmatisering  312523  20821


**Hvad blev slået sammen?** For hvert lemma: hvor mange forskellige ordformer dækker det over?

In [29]:
sammen_s = (ord
            .groupby("form")["for_lemma"]
            .agg(ordformer="nunique", eksempler="unique")
            .sort_values("ordformer", ascending=False)
            .reset_index())

**Kig efter i viewer:** Øverst står de lemmaer, der har samlet flest bøjninger, og det er typisk rigtigt: *forslag*,
*forslaget*, *forslagene*. Rul ned og kig efter to slags fejl:

- ord, der er slået sammen, men ikke burde være det,
- lemmaer, der ikke er rigtige danske ord, som *skatt* i mikroeksemplet.

Lemmaer, der er identiske med ordformen, er enten allerede i grundform eller ukendte for
modellen. I mikroeksemplet fik *Ordføreren* sig selv som lemma. Søg efter `ordføreren` i
viewer og se, om det samme sker i debatterne.

In [30]:
uaendret = ord[ord["for_lemma"] == ord["form"]]["form"].value_counts().reset_index()

**Kig efter i viewer:** Hvilke af disse ord er i bøjet form og burde have fået et andet lemma?

**Konsekvens:** Lemmatisering er mere præcis end stemming, men fejlene er **uforudsigelige**.
En stemmer laver den samme fejl hver gang, og man kan læse reglen. Modellen er trænet på
dansk nyhedstekst, og referater af debatter i Folketinget er ikke nyhedstekst.

### 5.6 Ordklasser: behold kun navneord, egennavne og tillægsord

Denne operation er ikke blandt Denny & Spirlings syv. I stedet for at fjerne ord fra en liste
**beholder** vi bestemte ordklasser, præcis som i lektion 6.

In [31]:
behold = ["NOUN", "PROPN", "ADJ"]

fjernet_pos = (ord[~ord["ordklasse"].isin(behold)]
               .groupby("ordklasse")["form"]
               .value_counts()
               .groupby(level=0)
               .head(10)
               .reset_index())

print(ord[~ord["ordklasse"].isin(behold)]["ordklasse"].value_counts())

ordklasse
VERB     74747
ADV      30302
SCONJ     2602
X         1796
DET       1391
ADP       1201
INTJ      1109
AUX        920
PRON       561
CCONJ      322
NUM        252
SYM         12
PUNCT        3
Name: count, dtype: int64


In [32]:
ord = ord[ord["ordklasse"].isin(behold)]

tael("+ kun NOUN, PROPN, ADJ", ord["form"])

                     trin  tokens  typer
0         Rå spaCy-tokens  922388  27535
1         + små bogstaver  922388  26203
2     + fjern tegnsætning  802536  26179
3             + fjern tal  776878  25664
4         + fjern stopord  312523  25452
5         + lemmatisering  312523  20821
6  + kun NOUN, PROPN, ADJ  197305  17284


**Kig efter i viewer:** De 10 hyppigste ord i hver fjernet ordklasse. Kig på `VERB`. Hvilke af udsagnsordene siger
noget om, hvad partierne mener: *støtte*, *stemme*, *afvise*, *foreslå*?

**Konsekvens:** I jobopslagene i lektion 6 var udsagnsordene næsten ens i alle opslag. I debatter er de ikke. Det er istedet udsagnsordene, der udtrykker holdning. Efter dette trin kan teksterne stadig fortælle, **hvad** der tales om, men ikke **hvad man mener** om det.

Om det er et problem, afhænger af spørgsmålet. Vil vi finde debatternes politikområde, som i klyngeanalysen, er emnet måske netop det, vi vil have. Vil vi måle partiernes holdninger, har vi smidt det vigtigste væk.

### 5.7 I: fjern sjældne ord

Som i lektion 6 beholder vi kun ord, der står i mindst 5 forskellige debatter. Først tæller vi, i hvor mange debatter hvert ord optræder (dokumentfrekvens).

In [33]:
dokfrekvens = ord.drop_duplicates(["id", "form"])["form"].value_counts()

print("Typer der kun står i én debat:", (dokfrekvens == 1).sum(), "af", len(dokfrekvens))

Typer der kun står i én debat: 11137 af 17284


In [34]:
almindelige = dokfrekvens[dokfrekvens >= 5].index

fjernet_i = ord.loc[~ord["form"].isin(almindelige), "form"].value_counts().reset_index()

ord = ord[ord["form"].isin(almindelige)]

tael("+ fjern sjældne (df < 5)", ord["form"])

                       trin  tokens  typer
0           Rå spaCy-tokens  922388  27535
1           + små bogstaver  922388  26203
2       + fjern tegnsætning  802536  26179
3               + fjern tal  776878  25664
4           + fjern stopord  312523  25452
5           + lemmatisering  312523  20821
6    + kun NOUN, PROPN, ADJ  197305  17284
7  + fjern sjældne (df < 5)  157449   2264


Det modsatte mønster af stopord, hvor **mange typer** forsvinder, men **få tokens**. Tabellen
bliver meget smallere, teksten bliver næsten ikke kortere.

**Kig efter i viewer:** Hvad slags ord er sjældne? Navne på personer, virksomheder og steder, fagudtryk,
stavefejl? Er der nogen, der kunne være interessante for et bestemt politikområde?

**Konsekvens:** Grænsen på 5 debatter er relativ til korpussets størrelse. I et udsnit på 100 debatter er det 5%. I hele korpusset på over 3.000 debatter er det under 0,2%. Samme regel giver et andet vokabular, når korpusset ændrer sig.

### 5.8 Hele kæden

Samme tabel som i lektion 6, nu for spaCy på Folketingets debatter. Vi tilføjer, hvor stor en
andel af udgangspunktet der er tilbage.

In [35]:
kaede["tokens_pct"] = (kaede["tokens"] / kaede["tokens"].iloc[0] * 100).round(1)
kaede["typer_pct"] = (kaede["typer"] / kaede["typer"].iloc[0] * 100).round(1)

print(kaede)

                       trin  tokens  typer  tokens_pct  typer_pct
0           Rå spaCy-tokens  922388  27535       100.0      100.0
1           + små bogstaver  922388  26203       100.0       95.2
2       + fjern tegnsætning  802536  26179        87.0       95.1
3               + fjern tal  776878  25664        84.2       93.2
4           + fjern stopord  312523  25452        33.9       92.4
5           + lemmatisering  312523  20821        33.9       75.6
6    + kun NOUN, PROPN, ADJ  197305  17284        21.4       62.8
7  + fjern sjældne (df < 5)  157449   2264        17.1        8.2


**Kig efter:** Hvilke trin fjerner **tokens**, og hvilke fjerner **typer**? 

---

## 6. Operation 3: ord, der hører sammen

I Folketinget blicer partinavnene *Dansk Folkeparti* til *dansk* og *folkeparti*, og så er partiet blevet
til et tillægsord, der også bruges om alt andet dansk.

spaCy har et færdigt trin, `merge_entities`, der slår hver navngiven enhed sammen til ét token.
Vi laver en ny pipeline med trinnet til sidst.

In [36]:
nlp_navne = spacy.load("da_core_news_sm")
nlp_navne.add_pipe("merge_entities")

print(nlp_navne.pipe_names)

['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'ner', 'merge_entities']


In [37]:
mikro_navne = tokentabel(nlp_navne(" ".join(mikro)))

print(mikro_navne.loc[mikro_navne["enhed"] != "", ["tekst", "lemma", "enhed"]])

               tekst             lemma enhed
9             danske             dansk  MISC
52           Venstre           venstre   ORG
54  Dansk Folkeparti  dansk folkeparti   ORG
59      Enhedslisten      Enhedslisten   PER


Nu er *Dansk Folkeparti* ét token. Vi kører udsnittet igennem igen og finder de navne, der
består af mere end ét ord, altså dem, der ville være blevet splittet.

In [38]:
udsnit["doc_navne"] = list(nlp_navne.pipe(udsnit["tekst"], batch_size=50))

tabeller = udsnit.set_index("id")["doc_navne"].map(tokentabel)
navne = pd.concat(tabeller.to_dict(), names=["id", "nr"]).reset_index()

flerordsnavne = (navne[navne["tekst"].str.contains(" ") & (navne["enhed"] != "")]
                 .groupby("enhed")["tekst"]
                 .value_counts()
                 .reset_index())

**Kig efter i viewer:** Filtrér på `ORG` og på `PER`. Hvilke partier, ministerier og personer ville være blevet
splittet i enkelte ord uden dette trin? Er der nogen, der er slået forkert sammen?

**Konsekvens:** Uden trinnet tælles *dansk* i *Dansk Folkeparti* med de almindelige
forekomster af tillægsordet. Med trinnet bevares partiet, men nu afhænger det af, om modellen
finder navnet. I mikroeksemplet blev *Enhedslisten* fundet, men som en person.

---

## 7. Samlet: den opdaterede model og hele kæden

Indtil nu har vi taget ét trin ad gangen for at se, hvad der forsvinder. Her samler vi det hele i to funktioner:

1. `byg_model()` henter modellen og tilføjer alle vores rettelser.
2. `forbehandl()` kører hele kæden ud fra en ordbog med valg.

Ordbogen med valg er Denny & Spirlings pointe i praksis, hvor hvert valg er en "kontakt", der kan slås til eller fra, og alle valg står samlet ét sted, så de kan skrives ind i metodeafsnittet.

### 7.1 Den opdaterede model

Alle rettelser fra trin 2.1, plus partinavnene som organisationer og `merge_entities` fra trin 6, så navne på flere ord bliver ét token.

Dette er en **basisskabelon**, der kan udvides til ens specifikke behov. 

In [40]:
def byg_model():
    model = spacy.load("da_core_news_sm")

    # Navngivne enheder: partierne er altid organisationer
    partier = ["Socialdemokratiet", "Venstre", "Dansk Folkeparti", "Radikale Venstre",
               "Socialistisk Folkeparti", "SF", "Enhedslisten", "Det Konservative Folkeparti",
               "Konservative", "Liberal Alliance", "Alternativet", "Nye Borgerlige",
               "Kristendemokraterne", "Moderaterne", "Danmarksdemokraterne"]
    regler = pd.DataFrame({"label": "ORG", "pattern": partier})
    ruler = model.add_pipe("entity_ruler", before="ner")
    ruler.add_patterns(regler.to_dict("records"))

    # Lemmaer, som modellen tager fejl af
    lemmaer = {"skatten": "skat", "ordføreren": "ordfører"}
    ar = model.get_pipe("attribute_ruler")
    for ordform, lemma in lemmaer.items():
        ar.add([[{"LOWER": ordform}]], {"LEMMA": lemma})

    # Egenskaber i ordforrådet
    model.vocab["§"].is_punct = False
    for form in ["tre", "Tre"]:
        model.vocab[form].like_num = False
    for form in ["ikke", "Ikke", "imod", "Imod"]:
        model.vocab[form].is_stop = False

    # Navne på flere ord bliver ét token
    model.add_pipe("merge_entities")

    return model

In [41]:
nlp_final = byg_model()

print(nlp_final.pipe_names)

['tok2vec', 'morphologizer', 'parser', 'lemmatizer', 'attribute_ruler', 'entity_ruler', 'ner', 'merge_entities']


Listen viser de to nye trin, med `entity_ruler` før `ner` og `merge_entities` til sidst.

`navne_som_skrevet` er et nyt valg, det betyder at personer, organisationer og steder beholder deres stavemåde og får hverken små bogstaver eller lemma. Så bliver *Venstre* (partiet) ikke til det samme som *venstre* (retningen), når modellen har fundet partiet.

In [42]:
valg = {
    "smaa_bogstaver":    True,
    "fjern_tegn":        True,
    "fjern_tal":         True,
    "fjern_stopord":     True,
    "lemma":             True,
    "ordklasser":        ["NOUN", "PROPN", "ADJ"],
    "navne_som_skrevet": True,
    "min_dokumenter":    5,
}

### 7.2 Hele kæden i én funktion

Hvert `if` er et af trinnene fra trin 5. Funktionen tager en tokentabel og ordbogen med valg og
giver den rensede tokentabel tilbage.

In [43]:
def preprocess_tekst(tokens, valg):
    t = tokens[~tokens["mellemrum"]].copy()

    if valg["fjern_tegn"]:
        t = t[~t["tegn"]]
    if valg["fjern_tal"]:
        t = t[~t["tal"]]
    if valg["fjern_stopord"]:
        t = t[~t["stopord"]]
    if valg["ordklasser"] is not None:
        t = t[t["ordklasse"].isin(valg["ordklasser"])]

    t["form"] = t["lemma"] if valg["lemma"] else t["tekst"]
    if valg["smaa_bogstaver"]:
        t["form"] = t["form"].str.lower()
    if valg["navne_som_skrevet"]:
        er_navn = t["enhed"].isin(["PER", "ORG", "LOC"])
        t["form"] = t["form"].where(~er_navn, t["tekst"])

    dokfrekvens = t.drop_duplicates(["id", "form"])["form"].value_counts()
    t = t[t["form"].isin(dokfrekvens[dokfrekvens >= valg["min_dokumenter"]].index)]

    return t

`.where(~er_navn, t["tekst"])` betyder: behold `form`, hvor tokenet ikke er et navn, og brug
den oprindelige tekst, hvor det er.

### 7.3 Test på mikroeksemplet

Mikroeksemplet er kun ét dokument, så vi sætter `min_dokumenter` til 1. `dict(valg, ...)`
laver en kopi af valgene med én ændring.

In [44]:
mikro_final = tokentabel(nlp_final(" ".join(mikro)))
mikro_final["id"] = 1

print(preprocess_tekst(mikro_final, dict(valg, min_dokumenter=1))["form"].tolist())

['regering', 'skat', 'arbejde', 'dansk', 'lønmodtager', 'stk.', 'milliard', 'krone', 'minister', 'gang', 'l', 'klima-venlig', 'forslag', 'Venstre', 'Dansk Folkeparti', 'forslag', 'Enhedslisten', 'ordfører', 'venstre', 'sal']


### 7.4 Kør på udsnittet

Hele kæden, fra tekst til dokument-term-matrix. For at køre på hele korpusset udskiftes
`udsnit` med `lovforslag`. Se trin 3.2 for, hvor lang tid det tager.

In [45]:
udsnit["doc_final"] = list(nlp_final.pipe(udsnit["tekst"], batch_size=50))

tabeller = udsnit.set_index("id")["doc_final"].map(tokentabel)
ord = pd.concat(tabeller.to_dict(), names=["id", "nr"]).reset_index()
ord = ord.merge(udsnit[["id", "omraade"]], on="id", how="left")

renset = preprocess_tekst(ord, valg)

dtm = pd.crosstab(renset["id"], renset["form"])

print(dtm.shape)

(100, 2172)


**Kig efter i viewer:** Én række per debat, én kolonne per ord. Find kolonnerne med partinavne.
Står *Venstre* og *venstre* som to kolonner?

### 7.5 Er resultatet robust?

Med valgene samlet i én ordbog kan vi køre flere kombinationer og se, hvor meget de flytter
vokabularet. Det er idéen bag Denny & Spirlings `preText`.

In [46]:
varianter = {
    "som ovenfor":           valg,
    "alle ordklasser":       dict(valg, ordklasser=None),
    "med nægtelser (ADV)":   dict(valg, ordklasser=["NOUN", "PROPN", "ADJ", "ADV"]),
    "uden lemma":            dict(valg, lemma=False),
    "ingen stopord fjernet": dict(valg, fjern_stopord=False),
}

robust = pd.DataFrame({navn: preprocess_tekst(ord, v)["form"].agg(["size", "nunique"])
                       for navn, v in varianter.items()}).T
robust.columns = ["tokens", "typer"]

print(robust)

                       tokens  typer
som ovenfor            149863   2172
alle ordklasser        269890   3298
med nægtelser (ADV)    189631   2459
uden lemma             145012   2746
ingen stopord fjernet  159673   2194


**Til diskussion:** Hvilke valg ændrer vokabularet mest? 

Næste skridt er at køre PCA og k-means på hver variant og se, om klyngerne ændrer sig. Hvis de gør, afhænger konklusionen af forbehandlingen og ikke kun af teksterne.

---

## 7. Regler over for model på de samme tekster

Til sidst regelkæden fra lektion 6 på præcis de samme 100 tekster, så vi kan sammenligne.
For at gøre sammenligningen fair bruger vi spaCys stopordsliste i begge.

In [47]:
regelkaede = pd.DataFrame(columns=["trin", "tokens", "typer"])

def tael_regel(trin, former):
    regelkaede.loc[len(regelkaede)] = [trin, len(former), former.nunique()]

r = udsnit.set_index("id")["tekst"].str.split().explode()
tael_regel("Råt .split()", r)

r = r.str.lower()
tael_regel("+ små bogstaver", r)

tegn = string.punctuation + "«»…–—"
r = r.str.strip(tegn)
r = r[r != ""]
tael_regel("+ fjern tegnsætning", r)

r = r[~r.str.contains(r"\d")]
tael_regel("+ fjern tal", r)

r = r[~r.isin(stopord)]
tael_regel("+ fjern stopord", r)

par = r.rename("ord").reset_index().drop_duplicates()
df_r = par["ord"].value_counts()
r = r[r.isin(df_r[df_r >= 5].index)]
tael_regel("+ fjern sjældne (df < 5)", r)

print(regelkaede)

                       trin  tokens  typer
0              Råt .split()  805378  42450
1           + små bogstaver  805378  41041
2       + fjern tegnsætning  802598  26035
3               + fjern tal  797775  25357
4           + fjern stopord  313376  25142
5  + fjern sjældne (df < 5)  259723   4576


Regelkæden har hverken lemmatisering eller ordklasser, så de to tabeller har ikke de samme
trin. Men start og slut kan sammenlignes:

---

## 8. Konsekvensen

Samme tre punkter som i slides fra lektion 6, nu med Folketinget som eksempel:

- **Tid.** Reglerne kørte på sekunder. Se trin 3.2 for, hvor lang tid spaCy ville bruge på
  hele korpusset.
- **Sproget.** Modellen er trænet på dansk nyhedstekst. Debatter i Folketinget er talesprog,
  skrevet ned, med partinavne, paragraffer og lovforslagsnumre. Fejlene i 5.5 og 6 kommer
  derfra.
- **Gennemsigtighed.** Stopordslisten kan læses (trin 5.4). Hvorfor `§` er tegnsætning, `tre`
  er et tal, og *Enhedslisten* er en person, kan ikke læses nogen steder.

### Valgene, der hører i metodeafsnittet

| Trin | Valg | Hvad forsvinder i en debat i Folketinget |
|---|---|---|
| L | små bogstaver | forskellen på partier og almindelige ord (*Venstre*/*venstre*) |
| P | `is_punct` | `§` |
| N | `like_num` | paragraffer, årstal, beløb, lovforslagsnumre og tal skrevet med bogstaver |
| W | spaCys stopordsliste | nægtelser: *ikke*, *ingen*, *aldrig*, *imod* |
| S | lemmatisering | bøjninger, og nogle gange ord, modellen ikke kender |
| — | kun NOUN, PROPN, ADJ | udsagnsord, og dermed det meste af holdningen |
| I | df ≥ 5 | navne, fagudtryk og sjældne emner, afhængigt af korpussets størrelse |
| 3 | `merge_entities` | *(bevarer)* flerordsnavne, hvis modellen finder dem |

<br>                   
<br>                   

> Ingen af valgene er forkerte i sig selv. Men hvert af dem er en beslutning om, hvad
> analysen *kan* finde, og den bør træffes med forskningsspørgsmålet for øje, ikke fordi
> det stod i en tutorial.